# Model B: Feature Engineering

## Objective

This notebook prepares the market features required for the unsupervised
regime-detection component of the project.

Using the previously cleaned and merged S&P 500 and CBOE VIX daily dataset,
the notebook transforms the raw market variables into features designed to
capture different dimensions of market conditions.

The feature-engineering process includes:

- calculating daily S&P 500 log returns;
- constructing VIX level and daily VIX change measures;
- calculating 5-, 10-, and 20-day realized volatility;
- calculating 5-, 10-, and 20-day rolling return skewness;
- calculating 5-, 10-, and 20-day rolling return kurtosis;
- evaluating the variability and correlation of the candidate features; and
- selecting a parsimonious feature set for the primary Hidden Markov Model.

The purpose of this notebook is therefore to produce a reproducible,
model-ready feature dataset. Hidden Markov Model estimation and regime
identification are performed in the subsequent Model B modeling notebook.


## Role in the Overall Project

The overall project investigates whether financial markets exhibit distinct
underlying volatility regimes and whether information about those regimes can
improve short-term volatility forecasting.

The project consists of two connected modeling components:

### Model B — Unsupervised Regime Detection

Model B uses unsupervised learning to identify latent market regimes from
S&P 500 and VIX market information without using predefined regime labels.

A Gaussian Hidden Markov Model (HMM) serves as the primary regime-detection
method, while K-means clustering provides a simpler comparison approach.

The features produced in this notebook provide the inputs required for these
models. The resulting latent states will subsequently be interpreted using
their market characteristics and described as regimes such as calm,
transitional, and stressed.

Model B will also generate regime information, including estimated regime
membership probabilities, that can be used in the supervised forecasting
component of the project.


### Model A — Supervised Volatility Forecasting

Model A evaluates whether the regime information generated by Model B provides
additional predictive value for short-term market volatility.

The supervised model predicts forward 5-day realized volatility using
observable market indicators. Two specifications will be compared:

1. a model using conventional market features; and
2. a model using the same market features together with regime information
   generated by Model B.

This comparison allows the project to evaluate whether incorporating latent
market-state information improves volatility forecasting beyond observable
market indicators alone.

## Outputs of This Notebook

This notebook produces two Parquet datasets for subsequent analysis.

### 1. Full Engineered Feature Dataset
`market_full_features_2000_2025.parquet`

Contains the complete set of engineered 5-, 10-, and 20-day candidate
features. This dataset is retained for reproducibility, alternative model
specifications, and robustness analysis.


### 2. Selected HMM Feature Dataset
`market_hmm_features_2000_2025.parquet`

Contains the features selected for the primary HMM specification:

- VIX Level
- VIX Change
- 20-Day Realized Volatility
- 20-Day Return Skewness
- 20-Day Return Kurtosis

The dataset also retains `Date` and `SP500_Close` for temporal alignment,
interpretation, and later visualization of the inferred market regimes.

Both output datasets contain complete observations from February 1, 2000
through December 31, 2025.

The selected feature dataset will serve as the primary input to the subsequent
Model B regime-detection notebook. The regime information produced by that
model will then be incorporated into Model A to test whether latent market
regimes improve short-term volatility forecasts.

-------

##1. Environment Setup

The primary unsupervised model in this analysis is a Gaussian Hidden Markov
Model (HMM).

Google Colab does not always include the `hmmlearn` package by default.
Therefore, the package is installed before beginning the analysis.

The `-q` option performs the installation in quiet mode so that unnecessary
installation messages are not displayed in the notebook.


In [1]:
!pip install hmmlearn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 1.5 MB/s eta 0:00:00


## 2. Import Required Libraries

The following Python libraries are used throughout the analysis:

- **NumPy (`numpy`)** provides numerical operations and mathematical functions.
- **Pandas (`pandas`)** is used for loading, manipulating, and summarizing the
  market dataset.
- **Matplotlib (`matplotlib.pyplot`)** is used to create visualizations of the
  market data and estimated regimes.
- **StandardScaler** standardizes model features so that they have approximately
  zero mean and unit variance.
- **KMeans** provides the comparison clustering model.
- **silhouette_score** measures how well observations are separated across the
  K-means clusters.
- **GaussianHMM** provides the Gaussian Hidden Markov Model used to identify
  latent market regimes.

Warnings are suppressed to keep the notebook output easier to read.

Pandas display settings are also adjusted so that all columns are visible and
numerical results are displayed consistently.

In [2]:
# Data manipulation libraries : used for feature engineering
import numpy as np
import pandas as pd

# Visualization library : used to plot and visualize model results
import matplotlib.pyplot as plt

# Pre-processing library : Transforms modelfeatures to comparable scales before fitting the HMM and K-Means models
from sklearn.preprocessing import StandardScaler

# Clustering algorithm : K-Means clustering algorithm that will be used as a benchmark for the unsupervised model
from sklearn.cluster import KMeans

# Evaluation metric : used to evaluate how clearly the k-means observations are seperated into clusters
from sklearn.metrics import silhouette_score

# Unsupervised learning algorithm : Gaussian Hidden Markov Model, used to estimate latent market states.
from hmmlearn.hmm import GaussianHMM

# Misc imports : warning and display preferences.
import warnings
warnings.filterwarnings("ignore") # suppress non-critical warnings

pd.set_option("display.max_columns", None) # displays each column when viewing dataframe
pd.set_option("display.float_format", lambda x: f"{x:,.4f}") # formats values to four decimal places to make results easier to read

## 3. Upload the Merged Market Dataset

The analysis uses the previously prepared `market_2000_2025.parquet` dataset.

This dataset combines daily S&P 500 and CBOE VIX observations on common trading
dates using an inner join. The merged dataset contains 6,539 observations
covering the period from January 2000 through December 2025.

Because the S&P 500 and VIX datasets have already been cleaned and merged,
this notebook begins with the combined dataset rather than repeating the
original data acquisition and merging process.

Parquet dataset has already been uploaded into the notebook environment.

In [6]:
# open and load file
df = pd.read_parquet("/content/market_2000_2025.parquet")

In [7]:
# lets take a look at "df"
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (6539, 11)


,Date,SP500_Open,SP500_High,SP500_Low,SP500_Close,SP500_Adj_Close,SP500_Volume,VIX_Open,VIX_High,VIX_Low,VIX_Close
0,2000-01-03,"1,469.2500","1,478.0000","1,438.3600","1,455.2200","1,455.2200",931800000,24.3600,26.1500,23.9800,24.2100
1,2000-01-04,"1,455.2200","1,455.2200","1,397.4301","1,399.4200","1,399.4200",1009000000,24.9400,27.1800,24.8000,27.0100
2,2000-01-05,"1,399.4200","1,413.2700","1,377.6801","1,402.1100","1,402.1100",1085500000,27.9800,29.0000,25.8500,26.4100
3,2000-01-06,"1,402.1100","1,411.9000","1,392.1000","1,403.4500","1,403.4500",1092300000,26.6800,26.7100,24.7000,25.7300
4,2000-01-07,"1,403.4500","1,441.4700","1,400.7300","1,441.4700","1,441.4700",1225200000,25.1400,25.1700,21.7200,21.7200


## 4. Inspect Dataset Structure

Before constructing the features used in the unsupervised learning models,
the structure and quality of the merged dataset are examined.

It is important to note that this dataset has already undergone initial data
preparation. The original S&P 500 and CBOE VIX datasets were checked for
missing values and duplicate dates and were subsequently merged using an
inner join on trading date. During this process, 28 dates that appeared in
the VIX dataset but not in the S&P 500 dataset were excluded. These additional
dates mainly corresponded to U.S. equity market closure days.

As a result, the dataset used in Model B contains only dates for which both
S&P 500 and VIX observations are available. The final merged dataset contains
6,539 daily observations covering the period from January 2000 through
December 2025.

Although these data-quality checks were performed during the initial data
preparation stage, they are repeated here to verify that the correct dataset
has been loaded and that its structure has been preserved before feature
engineering and model estimation.

The inspection verifies:

- the dimensions and variables contained in the dataset;
- the data type assigned to each variable;
- the available date range;
- whether any observations contain missing values;
- whether duplicate rows exist; and
- whether any trading dates appear more than once.

Maintaining a correctly ordered and consistently aligned time series is
particularly important for the Hidden Markov Model because the model uses
the sequence of observations to estimate latent market states and the
transitions between them.

In [9]:
# Health Checks
print("DATASET INFORMATION")
print("-" * 50)

# Display the structure, data types, and non-null observations
df.info()

# Display the available date range
print("\nDate range:")
print(df["Date"].min(), "to", df["Date"].max())

# Count missing observations in each variable
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing"))

# Check for duplicate observations and duplicate trading dates
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate dates:", df["Date"].duplicated().sum())

DATASET INFORMATION
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6539 entries, 0 to 6538
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Date             6539 non-null   datetime64[ns]
 1   SP500_Open       6539 non-null   float64       
 2   SP500_High       6539 non-null   float64       
 3   SP500_Low        6539 non-null   float64       
 4   SP500_Close      6539 non-null   float64       
 5   SP500_Adj_Close  6539 non-null   float64       
 6   SP500_Volume     6539 non-null   int64         
 7   VIX_Open         6539 non-null   float64       
 8   VIX_High         6539 non-null   float64       
 9   VIX_Low          6539 non-null   float64       
 10  VIX_Close        6539 non-null   float64       
dtypes: datetime64[ns](1), float64(9), int64(1)
memory usage: 562.1 KB

Date range:
2000-01-03 00:00:00 to 2025-12-31 00:00:00

Missi

,Missing
Date,0
SP500_Open,0
SP500_High,0
SP500_Low,0
SP500_Close,0
SP500_Adj_Close,0
SP500_Volume,0
VIX_Open,0
VIX_High,0
VIX_Low,0



Duplicate rows: 0
Duplicate dates: 0


## 5. Descriptive Statistics
Before constructing the features used for market regime detection, descriptive
statistics are examined for the numerical variables in the merged dataset.

This provides an overview of the scale, central tendency, and variability of
the S&P 500 and VIX variables before any transformations are applied. It also
serves as an additional check for potentially unusual values that may require
further investigation.

For each numerical variable, the summary reports the number of observations,
mean, standard deviation, minimum, maximum, and the 25th, 50th, and 75th
percentiles.

At this stage, these statistics describe the existing market variables only.
The derived features used by the Hidden Markov Model, including VIX changes,
realized volatility, skewness, and kurtosis, will be constructed and examined
separately in the feature-engineering stage.

In [10]:
# Generate descriptive statistics for the numerical variables
# This generates summary statistics for each numerical variable in the dataset, then transposes it so we can compare it accross the S&P 500 and VIX variables
summary_stats = df.describe().T

# Display the summary statistics
display(summary_stats)

,count,mean,min,25%,50%,75%,max,std
Date,6539,2012-12-31 14:04:31.968190976,2000-01-03 00:00:00,2006-07-05 12:00:00,2013-01-02 00:00:00,2019-07-01 12:00:00,2025-12-31 00:00:00,NaN
SP500_Open,"6,539.0000","2,268.6176",679.2800,"1,210.1250","1,527.2900","2,891.1650","6,936.0200","1,453.8762"
SP500_High,"6,539.0000","2,281.5469",695.2700,"1,217.2050","1,533.7400","2,899.4850","6,945.7700","1,460.5453"
SP500_Low,"6,539.0000","2,254.6831",666.7900,"1,202.7000","1,519.1200","2,876.7000","6,921.6001","1,446.6601"
SP500_Close,"6,539.0000","2,268.9515",676.5300,"1,210.2700","1,526.7500","2,888.0750","6,932.0498","1,454.1429"
SP500_Adj_Close,"6,539.0000","2,268.9515",676.5300,"1,210.2700","1,526.7500","2,888.0750","6,932.0498","1,454.1429"
SP500_Volume,"6,539.0000","3,418,314,009.7874",0.0000,"2,311,660,000.0000","3,529,430,000.0000","4,260,655,000.0000","11,456,230,000.0000","1,513,326,882.4533"
VIX_Open,"6,539.0000",19.9904,2.5800,14.0900,17.9100,23.3950,82.6900,8.4334
VIX_High,"6,539.0000",20.9094,9.3100,14.6700,18.7200,24.4400,89.5300,9.0212
VIX_Low,"6,539.0000",19.0632,8.5600,13.5450,17.1000,22.3750,72.7600,7.7889


Observation:

The descriptive statistics confirm that the merged dataset contains 6,539
daily observations spanning January 2000 through December 2025.

The S&P 500 variables reflect the substantial increase in the index level over
the study period. The closing price ranges from approximately 676.53 to
6,932.05, with an average of approximately 2,268.95. The adjusted closing
price is identical to the reported closing price throughout the summary.

The VIX exhibits considerably greater relative variability. The VIX closing
level has an average of approximately 19.84 and a median of 17.78, while its
maximum reaches 82.69. The large difference between typical VIX observations
and the observed maximum is consistent with the presence of periods of
substantially elevated market volatility within the sample.

S&P 500 trading volume also varies considerably across the sample and contains
a minimum value of zero. However, trading volume is not included among the
features selected for the initial regime-detection model, so this observation
does not directly affect the proposed HMM specification.

Overall, the descriptive statistics indicate substantial variation in both
equity market levels and market volatility across the sample. The raw variables
also operate on very different numerical scales. The model will therefore use
derived market characteristics rather than the raw OHLC variables, and the
selected features will later be standardized before model estimation.

## 6. Feature Engineering

The Hidden Markov Model will not be estimated directly from the raw S&P 500
OHLC and VIX OHLC variables. Instead, a smaller set of derived features is
constructed to represent different dimensions of current market conditions.

The initial HMM specification uses the following features:

- **VIX Level:** the daily VIX closing value, representing market-implied
  volatility.
- **VIX Change:** the day-to-day change in the VIX, capturing changes in
  market-implied uncertainty.
- **20-Day Realized Volatility:** the annualized standard deviation of recent
  S&P 500 daily log returns.
- **20-Day Return Skewness:** the asymmetry of S&P 500 returns over the
  preceding 20 trading days.
- **20-Day Return Kurtosis:** the degree of tail concentration in S&P 500
  returns over the preceding 20 trading days.

A common 20-trading-day trailing window is used for realized volatility,
skewness, and kurtosis. This represents approximately one trading month and
provides a consistent horizon for describing recent market behavior.

Daily S&P 500 log returns are first calculated because they provide the basis
for the three trailing return statistics.

### 6.1 S&P 500 Daily Log Returns

Daily S&P 500 log returns are calculated from consecutive closing prices.

A log return measures the continuously compounded percentage change between
two consecutive observations and is calculated as:

**Log Return(t) = ln[Price(t) / Price(t-1)]**

where:

- **Price(t)** is the S&P 500 closing price on the current trading day;
- **Price(t-1)** is the closing price on the previous trading day; and
- **ln** represents the natural logarithm.

Returns are used instead of the raw S&P 500 index level because the index level
contains a long-term upward trend, whereas returns describe relative daily
market movements.

The resulting return series will subsequently be used to calculate trailing
realized volatility, skewness, and kurtosis.

In [12]:
# Calculate the daily S&P 500 log return : we shift the S&P 500 closing price series down by one observation. Making sure to provide the previous trading day's closing price for each current observation.
# while also calculating the ratio between the current opening and closing and the previous trading days closing price.
# Then applying the natural logarithm to the ratio.
df["SP500_Log_Return"] = np.log(
    df["SP500_Close"] / df["SP500_Close"].shift(1)
)

# Display the first observations
# The first return value is expected to be missing since there is no previous trading day observation.
display(
    df[["Date", "SP500_Close", "SP500_Log_Return"]].head()
)

,Date,SP500_Close,SP500_Log_Return
0,2000-01-03,"1,455.2200",NaN
1,2000-01-04,"1,399.4200",-0.0391
2,2000-01-05,"1,402.1100",0.0019
3,2000-01-06,"1,403.4500",0.0010
4,2000-01-07,"1,441.4700",0.0267


### 6.2 Daily Change in the VIX

The VIX level describes the current level of market-implied volatility, but
the direction and magnitude of its daily movement may provide additional
information about changing market conditions.

The daily VIX change is therefore calculated as:

**VIX Change(t) = VIX(t) - VIX(t-1)**

A positive value indicates that implied volatility increased relative to the
previous trading day, while a negative value indicates that implied volatility
declined.

Both the VIX level and its daily change will be retained because they capture
different information: the level represents the current volatility environment,
while the change captures short-term movement in market uncertainty.

In [13]:
# Retain the VIX closing level as a model feature
df["VIX_Level"] = df["VIX_Close"]

# Calculate the day-to-day change in the VIX
df["VIX_Change"] = df["VIX_Close"].diff()

# Display the first observations, first observation will be missing since there is no previous observation for comparison
display(
    df[["Date", "VIX_Level", "VIX_Change"]].head()
)

,Date,VIX_Level,VIX_Change
0,2000-01-03,24.2100,NaN
1,2000-01-04,27.0100,2.8000
2,2000-01-05,26.4100,-0.6000
3,2000-01-06,25.7300,-0.6800
4,2000-01-07,21.7200,-4.0100


### 6.3 Trailing Realized Volatility

Realized volatility is calculated over multiple trailing windows to capture
market variability across different short-term horizons.

Three windows are considered:

- **5 trading days**, representing approximately one trading week;
- **10 trading days**, representing approximately two trading weeks; and
- **20 trading days**, representing approximately one trading month.

For each window, realized volatility is calculated as the standard deviation
of daily S&P 500 log returns and annualized using the square root of 252.

Examining multiple horizons allows the analysis to determine whether very
recent volatility contains different information from the broader monthly
volatility environment.

The resulting features will be compared before fitting the Hidden Markov Model.
Highly correlated or redundant features will not necessarily all be included
in the final model specification.

In [17]:
# Calculate the rolling standard deviation of the previous 20 daily log returns
# 1. a moving window containing the most recent 20 daily S&P 500 log returns
# 2. calculate the standard deviation of the returns contained within each 5, 10, 20-day window. This measures the amount of variation in recent S&P 500 returns.
# 3. annualize the 5, 10, 20-day realized volatility
df["Realized_Vol_5D"] = (
    df["SP500_Log_Return"].rolling(window=5).std()
    * np.sqrt(252)
)

df["Realized_Vol_10D"] = (
    df["SP500_Log_Return"].rolling(window=10).std()
    * np.sqrt(252)
)

df["Realized_Vol_20D"] = (
    df["SP500_Log_Return"].rolling(window=20).std()
    * np.sqrt(252)
)

# Display the first 25 observations
# The first observations will contain missing values because a complete 20-return window is required before realized volatility can be calculated.
#These missing values are created by the rolling-window calculation and do not represent missing observations in the original market dataset.
display(
    df[
        [
            "Date",
            "SP500_Log_Return",
            "Realized_Vol_5D",
            "Realized_Vol_10D",
            "Realized_Vol_20D"
        ]
    ].head(25)
)

,Date,SP500_Log_Return,Realized_Vol_5D,Realized_Vol_10D,Realized_Vol_20D
0,2000-01-03,NaN,NaN,NaN,NaN
1,2000-01-04,-0.0391,NaN,NaN,NaN
2,2000-01-05,0.0019,NaN,NaN,NaN
3,2000-01-06,0.0010,NaN,NaN,NaN
4,2000-01-07,0.0267,NaN,NaN,NaN
5,2000-01-10,0.0111,0.3865,NaN,NaN
6,2000-01-11,-0.0131,0.2333,NaN,NaN
7,2000-01-12,-0.0044,0.2435,NaN,NaN
8,2000-01-13,0.0121,0.2468,NaN,NaN
9,2000-01-14,0.0106,0.1813,NaN,NaN


### 6.4 Trailing Return Skewness

Skewness measures the asymmetry of the recent S&P 500 return distribution.
Unlike realized volatility, which measures the magnitude of return variation,
skewness provides information about whether recent returns have been more
heavily concentrated toward positive or negative extremes.

Rolling skewness is calculated over three horizons:

- **5 trading days**, representing approximately one trading week;
- **10 trading days**, representing approximately two trading weeks; and
- **20 trading days**, representing approximately one trading month.

Under this measure:

- A value near **zero** indicates a relatively symmetric return distribution.
- A **negative value** indicates a more pronounced negative tail.
- A **positive value** indicates a more pronounced positive tail.

Using multiple horizons allows the analysis to examine whether the shape of
very recent returns differs from the broader monthly return distribution.

Shorter windows, particularly the 5-day window, contain fewer observations
and may therefore produce more variable estimates. The different windows will
be compared before determining which features should be retained in the final
Hidden Markov Model.

In [18]:
# Calculate skewness of S&P 500 log returns over a 20-day rolling window
# Negative values indicate that the recent return distribution has greater weight in its negative tail, while positive values indicate greater weight
# in its positive tail.
df["Return_Skew_5D"] = (
    df["SP500_Log_Return"]
    .rolling(window=5)
    .skew()
)

df["Return_Skew_10D"] = (
    df["SP500_Log_Return"]
    .rolling(window=10)
    .skew()
)

df["Return_Skew_20D"] = (
    df["SP500_Log_Return"]
    .rolling(window=20)
    .skew()
)
# Display observations around the first complete rolling window
# As with realized volatility, the first observations are expected to be
# missing because 5, 10, 20 valid daily returns are required before the first rolling
# skewness value can be calculated.
display(
    df[
        [
            "Date",
            "SP500_Log_Return",
            "Return_Skew_5D",
            "Return_Skew_10D",
            "Return_Skew_20D"
        ]
    ].head(25)
)

,Date,SP500_Log_Return,Return_Skew_5D,Return_Skew_10D,Return_Skew_20D
0,2000-01-03,NaN,NaN,NaN,NaN
1,2000-01-04,-0.0391,NaN,NaN,NaN
2,2000-01-05,0.0019,NaN,NaN,NaN
3,2000-01-06,0.0010,NaN,NaN,NaN
4,2000-01-07,0.0267,NaN,NaN,NaN
5,2000-01-10,0.0111,-1.2014,NaN,NaN
6,2000-01-11,-0.0131,0.4042,NaN,NaN
7,2000-01-12,-0.0044,0.6613,NaN,NaN
8,2000-01-13,0.0121,-0.0303,NaN,NaN
9,2000-01-14,0.0106,-0.9197,NaN,NaN


 Observation:

The rolling skewness features were successfully generated across the three
selected horizons. As expected, the first valid values become available after
5, 10, and 20 valid daily returns, respectively.

The initial observations also illustrate how the choice of rolling window
affects the behavior of the measure. The 5-day skewness series responds more
strongly to changes in recent returns, while the 10-day and 20-day measures
incorporate a longer return history and therefore tend to change more
gradually.

This difference represents a trade-off between responsiveness and stability.
A shorter window may identify abrupt changes in the return distribution more
quickly but may also be more sensitive to individual observations. Longer
windows provide more stable estimates but may respond more slowly to sudden
changes in market conditions.

No window is selected at this stage. The candidate measures will be evaluated
over the full sample before determining which features should be included in
the final HMM specification.

### 6.5 Trailing Return Kurtosis

Kurtosis provides information about the occurrence of unusually large returns
within the recent S&P 500 return distribution. It therefore complements
realized volatility and skewness by describing the extent to which recent
returns contain extreme observations.

Rolling kurtosis is calculated over the same three horizons used for realized
volatility and skewness:

- **5 trading days**, representing approximately one trading week;
- **10 trading days**, representing approximately two trading weeks; and
- **20 trading days**, representing approximately one trading month.

Pandas calculates **excess kurtosis**, where a normal distribution has an
excess kurtosis of approximately zero.

Under this convention:

- A value near **zero** indicates tail behavior broadly similar to a normal
  distribution.
- A **positive value** indicates heavier tails and a greater presence of
  extreme observations.
- A **negative value** indicates lighter tails relative to a normal
  distribution.

Using multiple horizons allows the analysis to compare the responsiveness and
stability of the kurtosis estimates. The shorter 5-day window may respond more
quickly to extreme market movements but is calculated from relatively few
observations and may therefore be less stable. The 10- and 20-day measures use
progressively more information and may provide more stable estimates.

All three horizons are initially retained as candidate features. Their
behavior and relationships with the other engineered features will be
evaluated before determining which variables should enter the final Hidden
Markov Model.

In [19]:
# Calculate excess kurtosis of S&P 500 log returns
# over a 5, 10, 20-day rolling window
df["Return_Kurt_5D"] = (
    df["SP500_Log_Return"]
    .rolling(window=5)
    .kurt()
)

df["Return_Kurt_10D"] = (
    df["SP500_Log_Return"]
    .rolling(window=10)
    .kurt()
)

df["Return_Kurt_20D"] = (
    df["SP500_Log_Return"]
    .rolling(window=20)
    .kurt()
)
# Display observations around the first complete rolling window
display(
    df[
        [
            "Date",
            "SP500_Log_Return",
            "Return_Kurt_5D",
            "Return_Kurt_10D",
            "Return_Kurt_20D"
        ]
    ].head(25)
)

,Date,SP500_Log_Return,Return_Kurt_5D,Return_Kurt_10D,Return_Kurt_20D
0,2000-01-03,NaN,NaN,NaN,NaN
1,2000-01-04,-0.0391,NaN,NaN,NaN
2,2000-01-05,0.0019,NaN,NaN,NaN
3,2000-01-06,0.0010,NaN,NaN,NaN
4,2000-01-07,0.0267,NaN,NaN,NaN
5,2000-01-10,0.0111,2.3723,NaN,NaN
6,2000-01-11,-0.0131,0.7275,NaN,NaN
7,2000-01-12,-0.0044,0.0158,NaN,NaN
8,2000-01-13,0.0121,-0.8947,NaN,NaN
9,2000-01-14,0.0106,-1.4830,NaN,NaN


Observation:

The rolling kurtosis measures were successfully calculated across the 5-, 10-,
and 20-day horizons. As expected, the first valid observations become available
after sufficient historical returns have accumulated for each rolling window.

The initial results show that the 5-day kurtosis measure changes considerably
more sharply than the longer-window measures. This is expected because the
statistic is calculated from only five recent returns, meaning that a single
large observation can have a substantial influence on the estimated shape of
the return distribution.

The 10-day measure appears less variable, while the 20-day measure provides a
broader representation of recent tail behavior. However, these initial
observations are not sufficient to determine which horizon should be used in
the HMM.

The full distributions, variability, and relationships among the candidate
features will therefore be evaluated before selecting the final model inputs.

The three kurtosis measures are retained as candidate features at this stage.
They will later be compared in terms of their variability and correlation
with other features before the final HMM specification is selected.

### 6.6 Review of the Candidate Feature Set

The feature-engineering stage has produced a set of candidate variables
representing different dimensions of recent market conditions.

The candidate features include:

- **VIX level**, representing the prevailing level of implied market volatility;
- **VIX change**, capturing daily changes in implied volatility;
- **5-, 10-, and 20-day realized volatility**, measuring the magnitude of
  recent S&P 500 return variation;
- **5-, 10-, and 20-day return skewness**, measuring asymmetry in recent
  returns; and
- **5-, 10-, and 20-day return kurtosis**, measuring the presence of extreme
  observations in recent returns.

At this stage, all engineered variables are retained as candidate features.
They will not automatically all be included in the Hidden Markov Model.

The next stage evaluates their distributions and correlations to determine
whether some variables are unstable or provide highly overlapping information.
This allows the final HMM specification to retain distinct information about
market conditions while avoiding unnecessary feature redundancy.

In [20]:
# stores the names of all variables currently being considered as potential inputs to the regime-detection model.
# Keeping the feature names in a Python list makes it easier to consistently select the same variables during exploratory analysis, missing-value checks,
# correlation analysis, and later model preparation.
candidate_features = [
    "VIX_Level",
    "VIX_Change",
    "Realized_Vol_5D",
    "Realized_Vol_10D",
    "Realized_Vol_20D",
    "Return_Skew_5D",
    "Return_Skew_10D",
    "Return_Skew_20D",
    "Return_Kurt_5D",
    "Return_Kurt_10D",
    "Return_Kurt_20D"
]

# Display the candidate features in one table
display(
    df[["Date"] + candidate_features].head(25)
)

,Date,VIX_Level,VIX_Change,Realized_Vol_5D,Realized_Vol_10D,Realized_Vol_20D,Return_Skew_5D,Return_Skew_10D,Return_Skew_20D,Return_Kurt_5D,Return_Kurt_10D,Return_Kurt_20D
0,2000-01-03,24.2100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-01-04,27.0100,2.8000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-01-05,26.4100,-0.6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-01-06,25.7300,-0.6800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-01-07,21.7200,-4.0100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2000-01-10,21.7100,-0.0100,0.3865,NaN,NaN,-1.2014,NaN,NaN,2.3723,NaN,NaN
6,2000-01-11,22.5000,0.7900,0.2333,NaN,NaN,0.4042,NaN,NaN,0.7275,NaN,NaN
7,2000-01-12,22.8400,0.3400,0.2435,NaN,NaN,0.6613,NaN,NaN,0.0158,NaN,NaN
8,2000-01-13,21.7100,-1.1300,0.2468,NaN,NaN,-0.0303,NaN,NaN,-0.8947,NaN,NaN
9,2000-01-14,19.6600,-2.0500,0.1813,NaN,NaN,-0.9197,NaN,NaN,-1.4830,NaN,NaN


### 7.1 Missing-Value Verification

The candidate features are first checked for missing observations.

The original merged market dataset was previously verified to contain no
missing values. However, the feature-engineering process intentionally
introduces missing values at the beginning of rolling variables because a
minimum number of historical returns is required before each statistic can
be calculated.

For example, a 20-day rolling measure requires 20 valid daily returns before
its first value can be calculated.

The purpose of this check is therefore to confirm that the missing observations
are consistent with the expected rolling-window calculations rather than
unexpected gaps in the underlying market data.

In [21]:
# Count missing observations in each candidate feature

missing_features = (
    df[candidate_features]
    .isna()
    .sum()
    .to_frame("Missing_Observations")
)

display(missing_features)

,Missing_Observations
VIX_Level,0
VIX_Change,1
Realized_Vol_5D,5
Realized_Vol_10D,10
Realized_Vol_20D,20
Return_Skew_5D,5
Return_Skew_10D,10
Return_Skew_20D,20
Return_Kurt_5D,5
Return_Kurt_10D,10


### 7.2 Construct the Complete Candidate Feature Dataset

The rolling calculations create missing values at the beginning of the dataset
because sufficient historical observations are not yet available to calculate
all candidate features.

Since the longest rolling window is 20 trading days, the earliest observations
do not contain a complete set of candidate features.

Rather than replacing these missing values, observations with incomplete
engineered features are removed. This avoids introducing artificial values
into the analysis.

A separate DataFrame named `features_df` is created so that the original
dataset remains unchanged. The S&P 500 closing price is also retained for
later visualization and interpretation of the detected market regimes, but it
is not currently treated as a candidate HMM input.

In [22]:
# Create a separate dataset for candidate feature evaluation

features_df = df[
    ["Date", "SP500_Close"] + candidate_features
].copy()

# Remove rows containing missing candidate features
features_df = features_df.dropna().reset_index(drop=True)

# Compare the original and complete feature datasets
print("Original observations:", len(df))
print("Complete feature observations:", len(features_df))
print("Observations removed:", len(df) - len(features_df))

print("\nFeature dataset date range:")
print(
    features_df["Date"].min(),
    "to",
    features_df["Date"].max()
)

# Display the first five complete observations
display(features_df.head())

Original observations: 6539
Complete feature observations: 6519
Observations removed: 20

Feature dataset date range:
2000-02-01 00:00:00 to 2025-12-31 00:00:00


,Date,SP500_Close,VIX_Level,VIX_Change,Realized_Vol_5D,Realized_Vol_10D,Realized_Vol_20D,Return_Skew_5D,Return_Skew_10D,Return_Skew_20D,Return_Kurt_5D,Return_Kurt_10D,Return_Kurt_20D
0,2000-02-01,"1,409.2800",23.4500,-1.5000,0.3115,0.2548,0.2636,-0.2534,-0.1560,-0.5230,0.3986,0.3141,0.4541
1,2000-02-02,"1,409.1200",23.1200,-0.3300,0.3095,0.2546,0.2233,-0.4654,-0.1424,-0.2181,0.8041,0.3277,0.5606
2,2000-02-03,"1,424.9700",22.0100,-1.1100,0.3137,0.2633,0.2266,-1.1525,-0.4509,-0.2915,1.8414,0.0363,0.3915
3,2000-02-04,"1,424.3700",21.5400,-0.4700,0.1649,0.2632,0.2266,0.7997,-0.5077,-0.2753,0.2310,0.0880,0.3819
4,2000-02-07,"1,424.2400",22.7900,1.2500,0.0965,0.2167,0.2048,0.6123,-0.6333,-0.5036,-3.2962,2.3529,0.8049


Observation:
The complete candidate feature dataset contains 6,519 observations covering
February 1, 2000 through December 31, 2025.

Only 20 observations were removed from the original 6,539 observations. These
correspond to the beginning of the sample, where insufficient historical
returns were available to calculate the longest 20-day rolling features.

No additional observations were lost, confirming that the missing values were
a consequence of the rolling feature construction rather than gaps in the
underlying merged market dataset.

The resulting `features_df` therefore provides a complete dataset for
evaluating the candidate HMM features.

### 7.3 Discriptive statistics

In [23]:
# Calculate descriptive statistics for all candidate features

candidate_summary = (
    features_df[candidate_features]
    .describe()
    .T
)

display(candidate_summary)

,count,mean,std,min,25%,50%,75%,max
VIX_Level,"6,519.0000",19.8278,8.3777,9.1400,13.9800,17.7500,23.2100,82.6900
VIX_Change,"6,519.0000",-0.0015,1.8147,-18.7100,-0.7200,-0.0900,0.5900,24.8600
Realized_Vol_5D,"6,519.0000",0.1555,0.1224,0.0084,0.0815,0.1247,0.1909,1.5229
Realized_Vol_10D,"6,519.0000",0.1605,0.1131,0.0232,0.0915,0.1319,0.1926,1.1967
Realized_Vol_20D,"6,519.0000",0.1635,0.1065,0.0328,0.0975,0.1370,0.1980,0.9790
Return_Skew_5D,"6,519.0000",0.0212,0.9275,-2.2091,-0.6143,0.0283,0.6590,2.2026
Return_Skew_10D,"6,519.0000",-0.0191,0.7491,-2.8308,-0.5155,-0.0219,0.4732,2.7891
Return_Skew_20D,"6,519.0000",-0.0919,0.6408,-3.1657,-0.4644,-0.0946,0.3090,2.5151
Return_Kurt_5D,"6,519.0000",0.0965,2.0054,-3.3285,-1.6557,0.0349,1.6739,4.8986
Return_Kurt_10D,"6,519.0000",0.2568,1.4770,-2.2077,-0.8010,-0.0810,0.9653,8.5845


Observation:

The descriptive statistics show meaningful differences across the candidate
rolling-window features.

Realized volatility becomes progressively more stable as the rolling window
increases. The standard deviation decreases from 0.1224 for the 5-day measure
to 0.1131 for the 10-day measure and 0.1065 for the 20-day measure. The maximum
also declines from 1.5229 for the 5-day measure to 0.9790 for the 20-day
measure. This suggests that shorter-window realized volatility is more
responsive to abrupt changes in market conditions, while the longer-window
measure provides a smoother representation of the prevailing volatility
environment.

A similar pattern appears in return skewness. The standard deviation declines
from 0.9275 for the 5-day measure to 0.7491 and 0.6408 for the 10- and 20-day
measures, respectively. This indicates that shorter-window skewness varies
more substantially over time. The mean 20-day skewness is slightly negative
(-0.0919), suggesting some asymmetry toward negative returns over the full
sample.

Kurtosis shows a somewhat different pattern. The 5-day measure has the largest
standard deviation (2.0054), indicating greater overall variability, while the
10- and 20-day measures both have standard deviations of approximately 1.48.
However, the maximum observed kurtosis increases with the window length,
reaching 12.4479 for the 20-day measure. Therefore, shorter windows appear more
variable overall, but longer windows can still record substantial tail
behavior during periods containing extreme market movements.

The VIX level also displays considerable variation, ranging from 9.14 to
82.69, while daily VIX changes range from -18.71 to 24.86. Together, these
features capture both the prevailing level of market-implied volatility and
short-term changes in market uncertainty.

These results suggest that the different rolling horizons do not behave
identically. However, descriptive statistics alone are insufficient to
determine whether the features provide distinct information. Correlation
analysis is therefore conducted next to assess potential redundancy among the
candidate variables.

### 1) Correlation

In [24]:
# Calculate the correlation matrix for all candidate features

correlation_matrix = (
    features_df[candidate_features]
    .corr()
)

# Display correlations rounded to two decimal places
display(correlation_matrix.round(2))

,VIX_Level,VIX_Change,Realized_Vol_5D,Realized_Vol_10D,Realized_Vol_20D,Return_Skew_5D,Return_Skew_10D,Return_Skew_20D,Return_Kurt_5D,Return_Kurt_10D,Return_Kurt_20D
VIX_Level,1.0000,0.1100,0.8000,0.8600,0.8700,-0.0200,-0.0000,0.0100,-0.0400,-0.0600,-0.0900
VIX_Change,0.1100,1.0000,-0.0400,-0.0300,-0.0400,-0.0200,-0.0400,-0.0600,-0.0000,0.0300,0.0300
Realized_Vol_5D,0.8000,-0.0400,1.0000,0.9100,0.8200,-0.0100,-0.0100,0.0100,-0.0100,-0.0600,-0.0400
Realized_Vol_10D,0.8600,-0.0300,0.9100,1.0000,0.9300,-0.0100,-0.0000,0.0200,-0.0400,-0.0400,-0.0600
Realized_Vol_20D,0.8700,-0.0400,0.8200,0.9300,1.0000,0.0100,0.0300,0.0500,-0.0400,-0.0600,-0.0600
Return_Skew_5D,-0.0200,-0.0200,-0.0100,-0.0100,0.0100,1.0000,0.4800,0.2600,0.0200,-0.0500,-0.0500
Return_Skew_10D,-0.0000,-0.0400,-0.0100,-0.0000,0.0300,0.4800,1.0000,0.5700,0.0100,-0.0400,-0.1300
Return_Skew_20D,0.0100,-0.0600,0.0100,0.0200,0.0500,0.2600,0.5700,1.0000,-0.0000,-0.0900,-0.2600
Return_Kurt_5D,-0.0400,-0.0000,-0.0100,-0.0400,-0.0400,0.0200,0.0100,-0.0000,1.0000,0.3200,0.1800
Return_Kurt_10D,-0.0600,0.0300,-0.0600,-0.0400,-0.0600,-0.0500,-0.0400,-0.0900,0.3200,1.0000,0.5500


Observation:
The correlation analysis reveals substantial differences in the degree of
overlap among the candidate feature groups.

The strongest relationships occur among the realized-volatility measures.
The 5- and 10-day measures have a correlation of 0.91, the 10- and 20-day
measures have a correlation of 0.93, and the 5- and 20-day measures have a
correlation of 0.82. This indicates that the three realized-volatility
variables contain considerable overlapping information.

The VIX level is also strongly positively correlated with realized volatility,
with correlations ranging from 0.80 to 0.87. This relationship is
economically intuitive because both variables reflect market volatility,
although they represent different concepts: the VIX reflects option-implied
market volatility, while realized volatility is calculated from observed
historical S&P 500 returns.

In contrast, VIX change has relatively weak correlations with the other
candidate variables. Its correlation with VIX level is only 0.11 and its
correlations with the realized-volatility measures are close to zero. This
suggests that daily changes in the VIX provide information about short-term
movements in market uncertainty that is distinct from the prevailing level
of volatility.

The rolling skewness measures exhibit moderate rather than very strong
correlations. The 5- and 10-day measures have a correlation of 0.48, the
10- and 20-day measures have a correlation of 0.57, and the 5- and 20-day
measures have a correlation of 0.26. Therefore, the different skewness
horizons appear to capture somewhat different aspects of recent return
asymmetry.

A similar pattern is observed for kurtosis. Correlations range from 0.18
between the 5- and 20-day measures to 0.55 between the 10- and 20-day
measures. The kurtosis measures are also only weakly correlated with most
of the volatility variables.

Overall, the results suggest particularly strong redundancy among the three
realized-volatility horizons, while the alternative skewness and kurtosis
windows contain more distinct information. However, retaining every available
horizon would increase the dimensionality and complexity of the HMM.

A more parsimonious feature specification should therefore be considered
before model estimation.

### 7.5 Selection of the HMM Feature Set

Based on the descriptive and correlation analyses, a parsimonious five-feature
specification is selected for the primary Hidden Markov Model.

The final features are:

- **VIX Level**
- **VIX Change**
- **20-Day Realized Volatility**
- **20-Day Return Skewness**
- **20-Day Return Kurtosis**

The 20-day horizon is retained for the rolling return-based measures to provide
a consistent approximately one-month representation of recent market
conditions. The descriptive analysis also showed that the shorter-window
measures, particularly the 5-day measures, tend to vary more substantially.

The correlation analysis provides an additional reason not to include every
rolling horizon simultaneously. In particular, the realized-volatility
measures are strongly correlated with one another, indicating substantial
overlap in the information they provide.

VIX level and 20-day realized volatility are also strongly correlated.
Nevertheless, both are retained because they represent conceptually different
dimensions of volatility: VIX reflects market-implied volatility, whereas
realized volatility measures volatility observed in historical S&P 500
returns.

The resulting feature set therefore represents volatility level, volatility
change, realized market variability, return asymmetry, and tail behavior
without including every overlapping rolling-window measure.

In [25]:
# Define the final features used in the primary HMM

hmm_features = [
    "VIX_Level",
    "VIX_Change",
    "Realized_Vol_20D",
    "Return_Skew_20D",
    "Return_Kurt_20D"
]

# Display the selected feature set
display(features_df[hmm_features].head())

,VIX_Level,VIX_Change,Realized_Vol_20D,Return_Skew_20D,Return_Kurt_20D
0,23.4500,-1.5000,0.2636,-0.5230,0.4541
1,23.1200,-0.3300,0.2233,-0.2181,0.5606
2,22.0100,-1.1100,0.2266,-0.2915,0.3915
3,21.5400,-0.4700,0.2266,-0.2753,0.3819
4,22.7900,1.2500,0.2048,-0.5036,0.8049


--------------

### 7.6 Save the Engineered Feature Datasets

Two versions of the engineered dataset are saved for reproducibility and
subsequent modeling.

The first dataset contains the complete set of candidate features developed
during feature engineering. Retaining this dataset allows alternative feature
specifications or robustness checks to be performed without repeating the
feature-engineering process.

The second dataset contains only the variables selected for the primary Hidden
Markov Model, together with the date and S&P 500 closing price. The closing
price is retained for later visualization of the detected regimes but is not
used as an HMM input.

Both datasets contain only complete observations from February 1, 2000 through
December 31, 2025.

In [26]:
# ---------------------------------------------------------
# Save 1: Full candidate feature dataset
# ---------------------------------------------------------

features_df.to_parquet(
    "/content/market_full_features_2000_2025.parquet",
    index=False
)


# ---------------------------------------------------------
# Save 2: Selected HMM feature dataset
# ---------------------------------------------------------

hmm_features_df = features_df[
    ["Date", "SP500_Close"] + hmm_features
].copy()

hmm_features_df.to_parquet(
    "/content/market_hmm_features_2000_2025.parquet",
    index=False
)


# Confirm that both datasets were saved successfully
print("Full feature dataset:")
print(features_df.shape)

print("\nSelected HMM feature dataset:")
print(hmm_features_df.shape)

display(hmm_features_df.head())

Full feature dataset:
(6519, 13)

Selected HMM feature dataset:
(6519, 7)


,Date,SP500_Close,VIX_Level,VIX_Change,Realized_Vol_20D,Return_Skew_20D,Return_Kurt_20D
0,2000-02-01,"1,409.2800",23.4500,-1.5000,0.2636,-0.5230,0.4541
1,2000-02-02,"1,409.1200",23.1200,-0.3300,0.2233,-0.2181,0.5606
2,2000-02-03,"1,424.9700",22.0100,-1.1100,0.2266,-0.2915,0.3915
3,2000-02-04,"1,424.3700",21.5400,-0.4700,0.2266,-0.2753,0.3819
4,2000-02-07,"1,424.2400",22.7900,1.2500,0.2048,-0.5036,0.8049
